# Step 01b — ERA5 MJJAS Download
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

This notebook downloads ERA5 atmospheric fields for **May–September (MJJAS)** to enable
the full Lee et al. (2013) BSISO preprocessing pipeline in Notebook 03:
- `u850` — zonal wind at 850 hPa
- `v850` — meridional wind at 850 hPa
- `OLR` — top net thermal radiation (proxy for outgoing longwave radiation)

**Why MJJAS instead of July-only?**  
The paper removes interannual variability by subtracting the **preceding 120-day running mean**
from each day. For July 1, this requires data back to ~March 3. By downloading MJJAS, we have
enough lead-in data (May + June = ~61 days) to compute a valid running mean for all July–September
days (the 120-day window is satisfied for days after ~day 61 of the season).

**Domain:** 60°E–160°E, 0°N–60°N, 2° resolution (same as Notebook 01)  
**Period:** May–September, 1979–2023 (45 years)  
**Output files:**
```
data/raw/u850_v850_MJJAS_YYYY_YYYY.nc   ← 5 year-chunks
data/raw/OLR_MJJAS_1979_2023.nc
```

---
⚠️ **Prerequisite:** CDS API key must be set up (same as Notebook 01).

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
RAW_DIR       = f'{PROJECT_DIR}/data/raw'

os.makedirs(RAW_DIR, exist_ok=True)
print('Google Drive mounted.')
print(f'Raw data directory: {RAW_DIR}')

## Cell 2 — Install CDS API Client

In [ ]:
!pip install cdsapi --quiet
import cdsapi
print('cdsapi ready.')

## Cell 3 — Set Up CDS API Credentials

In [ ]:
# ============================================================
# FILL IN YOUR PERSONAL ACCESS TOKEN HERE
# Find it at: https://cds.climate.copernicus.eu/ → Your profile
# ============================================================
CDS_API_KEY = 'bcc4c8e7-b5e7-44de-8f1d-d7414aab9a51'
# ============================================================

cdsapirc = f'url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n'
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(cdsapirc)

print('CDS credentials saved.')

try:
    client = cdsapi.Client(quiet=True)
    print('CDS API connection: OK')
except Exception as e:
    print(f'CDS API connection FAILED: {e}')

## Cell 4 — Download u850 + v850 for MJJAS

Downloads wind at 850 hPa for **May–September** in 5 year-chunks.  
Months requested: `05, 06, 07, 08, 09`  
**Estimated time:** 20–50 min total (CDS queue dependent)  
**Estimated file size:** ~150–200 MB total across all chunks

In [ ]:
import cdsapi
import os

# Re-init client (safe to call if already set up)
if not os.path.exists(os.path.expanduser('~/.cdsapirc')):
    raise RuntimeError('Run Cell 3 first to set up CDS credentials.')

client = cdsapi.Client()

MJJAS_MONTHS = ['05', '06', '07', '08', '09']
DAYS         = [f'{d:02d}' for d in range(1, 32)]  # CDS ignores invalid dates

year_chunks = [
    (1979, 1989),
    (1990, 1999),
    (2000, 2009),
    (2010, 2019),
    (2020, 2023),
]

for start_yr, end_yr in year_chunks:
    out_file = f'{RAW_DIR}/u850_v850_MJJAS_{start_yr}_{end_yr}.nc'

    if os.path.exists(out_file):
        size_mb = os.path.getsize(out_file) / 1e6
        print(f'[SKIP] {os.path.basename(out_file)} already exists ({size_mb:.1f} MB)')
        continue

    print(f'Downloading wind {start_yr}–{end_yr} MJJAS ... (do NOT close tab)')

    client.retrieve(
        'reanalysis-era5-pressure-levels',
        {
            'product_type'  : 'reanalysis',
            'variable'      : ['u_component_of_wind', 'v_component_of_wind'],
            'pressure_level': '850',
            'year'          : [str(y) for y in range(start_yr, end_yr + 1)],
            'month'         : MJJAS_MONTHS,
            'day'           : DAYS,
            'time'          : '12:00',
            'area'          : [60, 60, 0, 160],   # N, W, S, E
            'grid'          : [2.0, 2.0],
            'data_format'   : 'netcdf',
        },
        out_file
    )

    size_mb = os.path.getsize(out_file) / 1e6
    print(f'  Saved: {os.path.basename(out_file)}  ({size_mb:.1f} MB)\n')

print('All wind chunks done.')

## Cell 5 — Download OLR for MJJAS

Downloads top net thermal radiation (OLR proxy) for **May–September**, all years in one request.  
**Estimated time:** 10–25 min  
**Estimated file size:** ~60–80 MB

In [ ]:
out_olr = f'{RAW_DIR}/OLR_MJJAS_1979_2023.nc'

if os.path.exists(out_olr):
    size_mb = os.path.getsize(out_olr) / 1e6
    print(f'[SKIP] OLR_MJJAS_1979_2023.nc already exists ({size_mb:.1f} MB)')
else:
    print('Downloading OLR MJJAS 1979–2023 ... (do NOT close tab)')

    client.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable'    : 'top_net_thermal_radiation',
            'year'        : [str(y) for y in range(1979, 2024)],
            'month'       : MJJAS_MONTHS,
            'day'         : DAYS,
            'time'        : '12:00',
            'area'        : [60, 60, 0, 160],
            'grid'        : [2.0, 2.0],
            'data_format' : 'netcdf',
        },
        out_olr
    )

    size_mb = os.path.getsize(out_olr) / 1e6
    print(f'Saved: OLR_MJJAS_1979_2023.nc  ({size_mb:.1f} MB)')

## Cell 6 — Verify Downloads

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

print('=' * 60)
print('VERIFICATION REPORT — MJJAS ERA5 Downloads')
print('=' * 60)

# --- Wind chunks ---
print('\n[1] Wind files (u850 / v850)')
wind_files = sorted([
    f'{RAW_DIR}/{f}' for f in os.listdir(RAW_DIR)
    if f.startswith('u850_v850_MJJAS') and f.endswith('.nc')
])

all_wind = []
for wf in wind_files:
    ds = xr.open_dataset(wf).squeeze('pressure_level', drop=True)
    n_times = len(ds.valid_time)
    t_min   = str(ds.valid_time.values[0])[:10]
    t_max   = str(ds.valid_time.values[-1])[:10]
    size_mb = os.path.getsize(wf) / 1e6
    print(f'  {os.path.basename(wf):45s}  {n_times:4d} days  {t_min} → {t_max}  ({size_mb:.0f} MB)')
    all_wind.append(ds)

ds_wind_all = xr.concat(all_wind, dim='valid_time')
print(f'\n  Combined: {len(ds_wind_all.valid_time)} total time steps')
print(f'  Grid:     {len(ds_wind_all.latitude)} lat × {len(ds_wind_all.longitude)} lon')
print(f'  u range:  [{float(ds_wind_all["u"].min()):.2f}, {float(ds_wind_all["u"].max()):.2f}] m/s')

# Check months present
months_found = sorted(set(pd.DatetimeIndex(ds_wind_all.valid_time.values).month))
print(f'  Months present: {months_found}  (expected [5, 6, 7, 8, 9])')

# --- OLR ---
print('\n[2] OLR file')
ds_olr = xr.open_dataset(out_olr)
n_olr   = len(ds_olr.valid_time)
t_min   = str(ds_olr.valid_time.values[0])[:10]
t_max   = str(ds_olr.valid_time.values[-1])[:10]
size_mb = os.path.getsize(out_olr) / 1e6
print(f'  OLR_MJJAS_1979_2023.nc  {n_olr} days  {t_min} → {t_max}  ({size_mb:.0f} MB)')
print(f'  ttr range: [{float(ds_olr["ttr"].min()):.0f}, {float(ds_olr["ttr"].max()):.0f}] J/m²')

months_olr = sorted(set(pd.DatetimeIndex(ds_olr.valid_time.values).month))
print(f'  Months present: {months_olr}  (expected [5, 6, 7, 8, 9])')

# --- NaN check ---
nan_u   = int(np.isnan(ds_wind_all['u'].values).sum())
nan_ttr = int(np.isnan(ds_olr['ttr'].values).sum())
print(f'\n[3] NaN counts')
print(f'  u850: {nan_u}  (expected 0)')
print(f'  ttr:  {nan_ttr}  (expected 0)')

print('\nVerification complete.')

## Cell 7 — Quick Plot (Visual Sanity Check)

Plot one July day and one May day to confirm the seasonal contrast looks realistic.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

times = pd.DatetimeIndex(ds_wind_all.valid_time.values)

# Pick one May day and one July day
idx_may  = int(np.where(times.month == 5)[0][0])
idx_july = int(np.where(times.month == 7)[0][0])

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('MJJAS ERA5 Sanity Check — u850 and OLR', fontsize=14, fontweight='bold')

titles = [
    f'u850 — May sample ({str(times[idx_may])[:10]})',
    f'u850 — July sample ({str(times[idx_july])[:10]})',
]

time_dim_w = 'valid_time' if 'valid_time' in ds_wind_all.dims else 'time'
time_dim_o = 'valid_time' if 'valid_time' in ds_olr.dims else 'time'

olr_times = pd.DatetimeIndex(ds_olr.valid_time.values)
idx_may_o  = int(np.where(olr_times.month == 5)[0][0])
idx_july_o = int(np.where(olr_times.month == 7)[0][0])

u_may  = ds_wind_all['u'].isel(**{time_dim_w: idx_may}).squeeze().values
u_july = ds_wind_all['u'].isel(**{time_dim_w: idx_july}).squeeze().values
olr_may  = -ds_olr['ttr'].isel(**{time_dim_o: idx_may_o}).squeeze().values
olr_july = -ds_olr['ttr'].isel(**{time_dim_o: idx_july_o}).squeeze().values

extent = [60, 160, 0, 60]

for ax, data, title, cmap in zip(
    axes[0],
    [u_may, u_july],
    titles,
    ['RdBu_r', 'RdBu_r']
):
    im = ax.imshow(data, cmap=cmap, origin='upper', extent=extent, aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.colorbar(im, ax=ax, label='m/s')

for ax, data, title in zip(
    axes[1],
    [olr_may, olr_july],
    [f'OLR — May sample ({str(olr_times[idx_may_o])[:10]})',
     f'OLR — July sample ({str(olr_times[idx_july_o])[:10]})'],
):
    im = ax.imshow(data, cmap='RdBu_r', origin='upper', extent=extent, aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.colorbar(im, ax=ax, label='J/m²')

plt.tight_layout()
plt.show()

print('Expected: u850 should show strong westerly jet by July (stronger than May).')
print('Expected: OLR should show active convection (negative values) over Bay of Bengal in July.')

---
## Done!

If all cells ran successfully you should now have on Google Drive:

```
BSISO_SSL_Project/
└── data/
    └── raw/
        ├── u850_v850_MJJAS_1979_1989.nc
        ├── u850_v850_MJJAS_1990_1999.nc
        ├── u850_v850_MJJAS_2000_2009.nc
        ├── u850_v850_MJJAS_2010_2019.nc
        ├── u850_v850_MJJAS_2020_2023.nc
        └── OLR_MJJAS_1979_2023.nc
```

**Next step:** Run notebook `03_preprocessing.ipynb` (rewritten with Lee et al. 2013 method):
1. Remove annual cycle (climatological mean 1981–2010)
2. Remove interannual variability (120-day running mean)
3. Normalize by area-averaged std
4. Extract July–September model inputs

---
*DDCS Project | jh9141@nyu.edu*